# Kaggle Bootstrap — Sentiment Bias / Counterfactual Evaluation

Run this notebook top-to-bottom **at the start of every Kaggle session.**
It is idempotent: re-running it in a session that is already set up does nothing harmful.

## BEFORE you run anything — two settings in the right-hand panel

**1. Accelerator → `GPU T4 x2` or `GPU P100`.**
Kaggle requires **phone verification** before accelerators unlock. If the GPU options are
greyed out, that's why — verify your number in account settings first.

**2. Session options → Persistence → `Files Only`.**

> ⚠️ **This is the single most important setting on Kaggle.**
> `/kaggle/working` is **NOT** persistent by default. Kaggle staff have confirmed that for
> interactive sessions, output is *not* saved — when the session times out or is stopped,
> everything in `/kaggle/working` is lost and only your notebook *source code* survives.
>
> Two ways to make artefacts survive:
> - **Persistence → "Files Only"** (or "Variables and Files") — survives across sessions, best-effort.
> - **Save Version → Save & Run All (Commit)** — re-runs the notebook headless and permanently
>   stores everything in `/kaggle/working` at the end of the run. Use this after any long training run.
>
> Persistence is "best effort": if the notebook crashes or output exceeds limits it can fail.
> **So: checkpoint often, and commit after every training run.**

### Kaggle limits (verified Sept 2026)
| | |
|---|---|
| GPU | P100 16 GB, or T4 ×2 (32 GB combined) |
| Weekly GPU quota | ~30 h (rolling), TPU ~20 h |
| Session cap | ~9–12 h GPU, 12 h CPU; idle sessions get an *"Are you still there?"* prompt |
| Storage | ~20 GB, in `/kaggle/working`, persistent only if you opt in |
| Background runs | **Save & Run All (Commit)** keeps running after you close the tab |

Note vs Colab: Kaggle's quota is **published and predictable** (30 h), Colab's is opaque
(~15–30 h and can silently drop you to CPU). Kaggle's storage is **smaller** (20 GB) and
**conditional**, Colab's Drive is large and always-on.

---
## Cell 1 — sanity check the platform and the GPU

In [ ]:
import os, shutil, subprocess

print('cwd                :', os.getcwd())
print('/kaggle/working    :', os.path.isdir('/kaggle/working'))
print('/kaggle/input      :', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'n/a')
print()
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print()
u = shutil.disk_usage('/kaggle/working')
print(f'/kaggle/working    : {u.total/1024**3:.1f} GB total, {u.free/1024**3:.1f} GB free')
print()
print(subprocess.run(['python','-V'], capture_output=True, text=True).stdout.strip())

**Expected:** a `Tesla T4` or `Tesla P100` line from `nvidia-smi`, ~20 GB free.
If `nvidia-smi` says *command not found* → you did not select an Accelerator.

⚠️ Neither T4 (sm_75) nor P100 (sm_70) supports **bf16**. That does not matter here —
the project trains in **fp32** deliberately, because the debiasing loss is a cosine
distance multiplied by λ up to 100, and bf16's 8 mantissa bits make that gradient noisy.
Do not switch on mixed precision to make training faster.

---
## Cell 2 — get the code

Pick **one** of 2A / 2B.

### 2A — from GitHub (recommended: you get git history too)

In [ ]:
REPO = 'https://github.com/YOUR-USERNAME/sentiment_bias_project.git'
DEST = '/kaggle/working/sentiment_bias_project'

if os.path.isdir(DEST):
    print('already present, pulling latest')
    !cd {DEST} && git pull --quiet
else:
    !git clone --quiet {REPO} {DEST}

!ls {DEST}

### 2B — from an uploaded Kaggle Dataset (if you don't want to use GitHub)

`Add Data → Your Datasets → New Dataset`, upload the project folder, then attach it.
`/kaggle/input` is **read-only**, so copy it out to `/kaggle/working` before use.

In [ ]:
SRC  = '/kaggle/input/sentiment-bias-project'          # <- whatever you named it
DEST = '/kaggle/working/sentiment_bias_project'

if not os.path.isdir(DEST):
    shutil.copytree(SRC, DEST, dirs_exist_ok=True)
    print('copied', SRC, '->', DEST)
else:
    print('already present at', DEST)

!ls {DEST}

---
## Cell 3 — install dependencies

`torch` is **preinstalled with CUDA** on Kaggle. Do not pip-install it — you will
overwrite the CUDA build with a CPU one and wonder why the GPU sits idle.

In [ ]:
%cd /kaggle/working/sentiment_bias_project
%pip install --quiet -r requirements.txt
print('done')

---
## Cell 4 — bootstrap and verify

`paths.bootstrap()` creates the directory tree under `/kaggle/working/sentiment_bias_project`
and reports your persistence regime.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/sentiment_bias_project')

from src import paths
root = paths.bootstrap(repo_url=None, verbose=True)

print()
print('platform      :', paths.PLATFORM)
print('project root  :', paths.ROOT)
print('disk          :', paths.disk_report())
print('project size  :', paths.project_size())

In [ ]:
%cd /kaggle/working/sentiment_bias_project
!python -m src.check_environment

**Expected on Kaggle:**
```
✅ PASS | 3. CUDA / GPU      | Tesla T4 | 15.1 GB VRAM total, ... | cc (7, 5)
⚠️ WARN | 10. Persistence    | KAGGLE PERSISTENCE UNVERIFIED - Kaggle: /kaggle/working persists ONLY if ...
✅ PASS | 11. Mixed precision| Tesla T4 | bf16_supported=False | T4 (sm_75): fp16 + Tensor Cores, NO bf16 | PROJECT TRAINS IN fp32 ...
 SUMMARY: 10 passed, 1 warning, 0 failed
```
Check 10 is a **WARN, not a FAIL**, on Kaggle — deliberately. The Persistence switch lives
in the notebook UI and is invisible from inside the VM, so the script cannot verify it.
**You must confirm it yourself.** Everything else must be PASS.

---
## Cell 5 — restore artefacts from a previous session

Run this only if you committed a previous version and want its models/results back.

**How:** `Add Data → Notebook Output Files → <your notebook>` → attach it. It mounts
read-only under `/kaggle/input/<slug>/`. Then copy what you need into `/kaggle/working`.

In [ ]:
# point this at wherever the previous run's output got mounted
PREV = '/kaggle/input/<your-notebook-slug>/sentiment_bias_project'

if os.path.isdir(PREV):
    for sub in ['models', 'results', 'plots', 'data']:
        src = os.path.join(PREV, sub)
        if os.path.isdir(src):
            shutil.copytree(src, f'/kaggle/working/sentiment_bias_project/{sub}',
                            dirs_exist_ok=True)
            print(f'restored {sub}/')
else:
    print(f'no previous output mounted at {PREV} - skipping (normal on a first run)')

---
## Cell 6 — END OF SESSION: save everything

Run this **before** you close the tab, then commit.

1. Run this cell.
2. **Save Version → Save & Run All (Commit)** → give it a name.
   Kaggle re-executes the notebook headless and permanently stores `/kaggle/working`.
3. Next session: attach that version's output in Cell 5.

Faster alternative for small artefacts: just download them.
`Notebook output pane → ⋮ → Download` — or list them below and grab them individually.

In [ ]:
root = paths.ROOT
print('=== artefacts you are about to commit ===')
total = 0
for sub in ['models', 'results', 'plots', 'data']:
    d = root / sub
    if not d.is_dir():
        continue
    files = [f for f in d.rglob('*') if f.is_file()]
    mb = sum(f.stat().st_size for f in files) / 1024**2
    total += mb
    print(f'  {sub:8s} {len(files):4d} files  {mb:8.1f} MB')
print(f'  {"TOTAL":8s} {total:26.1f} MB')
print()
if total > 15_000:
    print('WARNING: approaching the ~20 GB /kaggle/working quota.')
print('Now: Save Version -> Save & Run All (Commit)')

---
## Durable storage, if you want belt-and-braces

Kaggle's persistence is best-effort. For checkpoints you cannot afford to lose, push them
to a **private Kaggle Dataset** from inside the notebook using the Kaggle API. This survives
everything, including a deleted notebook.

One-time setup: Kaggle → account → *Create New API Token* → download `kaggle.json` →
upload it as a (private) Kaggle Dataset and attach it.

In [ ]:
# only needed if you want the API-push route
if os.path.isfile('/kaggle/input/kaggle-api/kaggle.json'):
    !mkdir -p ~/.kaggle
    !cp /kaggle/input/kaggle-api/kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !pip install --quiet kaggle

    # package the checkpoints as a dataset version and push
    !mkdir -p /kaggle/working/_push
    !cd {paths.ROOT}/models && zip -qr /kaggle/working/_push/models.zip .
    !cd /kaggle/working/_push && \
      printf '%s' '{"title":"sbp-checkpoints","id":"YOUR-USERNAME/sbp-checkpoints","resources":[{"path":"models.zip"}]}' > dataset-metadata.json
    # first time:  kaggle datasets create -p .     afterwards:  kaggle datasets version -p . -m "update"
    !cd /kaggle/working/_push && kaggle datasets version -p . -m "checkpoints" || \
      kaggle datasets create -p .
else:
    print('kaggle.json not attached - skipping. Cell 6 (Save & Run All) is enough for most runs.')